In [1]:
# =========================================================
# MÉTODOS ESTADÍSTICOS — RESPUESTA INSTITUCIONAL Y TENDENCIA
# TGEU Trans Murder Monitoring — Colombia
# =========================================================

# --- CELDA 1: Cargar datos y preparar variables base ---
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt

col = pd.read_csv('/content/tgeu_colombia_limpio.csv')

ACCION_IDENTIFICADA = [
    'caso en investigación', 'autor detenido', 'sospechoso identificado',
    'autor acusado', 'autor sentenciado', 'sospechoso detenido',
    'sospechoso declarado inocente', 'otro'
]
col['hubo_accion'] = col['Response from local authorities'].apply(
    lambda v: 1 if v in ACCION_IDENTIFICADA else 0
)
col['falta_dato_respuesta'] = col['Response from local authorities'].isna().astype(int)
col['es_travesti'] = (col['Gender identity or expression'] == 'travesti').astype(int)
col['año_centrado'] = col['Calendar year'] - col['Calendar year'].mean()

TOP4_CIUDADES = ['Bogotá', 'Medellín', 'Cali', 'Barranquilla']
col['es_capital_grande'] = col['ciudad_limpia'].isin(TOP4_CIUDADES).astype(int)

print(f"N = {len(col)}")

N = 306


In [2]:
# =========================================================
# MÉTODO 1 — PRUEBA DE FISHER: ¿travesti vs. mujer trans difieren
# de verdad en respuesta institucional, o es ruido de muestra chica?
# =========================================================

# --- CELDA 2 ---
MUJER_TRANS = 'mujer trans / trans femenina / trans femme / mujer transgénero'
sub = col[col['Gender identity or expression'].isin(['travesti', MUJER_TRANS])]
tabla = pd.crosstab(sub['Gender identity or expression'], sub['hubo_accion'])
print(tabla)

odds, p = fisher_exact(tabla)
print(f"\nPrueba de Fisher: OR = {odds:.3f}, p = {p:.6f}")
print("La diferencia es estadísticamente muy significativa: NO es ruido de")
print("muestra pequeña. Pero eso no basta para concluir discriminación por")
print("identidad — sigue el chequeo del confusor temporal en la Celda 3.")

hubo_accion                                           0   1
Gender identity or expression                              
mujer trans / trans femenina / trans femme / mu...  130  73
travesti                                             50   0

Prueba de Fisher: OR = 0.000, p = 0.000000
La diferencia es estadísticamente muy significativa: NO es ruido de
muestra pequeña. Pero eso no basta para concluir discriminación por
identidad — sigue el chequeo del confusor temporal en la Celda 3.


In [3]:
# --- CELDA 3: ¿Es un efecto de identidad o un efecto de época? ---
resumen_año = sub.groupby('Gender identity or expression')['Calendar year'].agg(['mean', 'min', 'max'])
print(resumen_año)
print("\nLos casos 'travesti' están concentrados en años tempranos (promedio")
print(f"{resumen_año.loc['travesti','mean']:.1f}); los de 'mujer trans' en años recientes")
print(f"(promedio {resumen_año.loc[MUJER_TRANS,'mean']:.1f}). Esto es un confusor real:")
print("ya sabemos (ver Método 3) que los casos recientes tienen más seguimiento")
print("documentado en general, sin importar la identidad de la víctima.")

                                                           mean   min   max
Gender identity or expression                                              
mujer trans / trans femenina / trans femme / mu...  2019.684729  2008  2025
travesti                                            2011.860000  2008  2018

Los casos 'travesti' están concentrados en años tempranos (promedio
2011.9); los de 'mujer trans' en años recientes
(promedio 2019.7). Esto es un confusor real:
ya sabemos (ver Método 3) que los casos recientes tienen más seguimiento
documentado en general, sin importar la identidad de la víctima.


In [4]:
# =========================================================
# MÉTODO 2 — ¿SE PUEDE AISLAR EL EFECTO DE 'TRAVESTI' CONTROLANDO
# POR AÑO? (se documenta el intento y su límite, no se esconde)
# =========================================================

# --- CELDA 4 ---
X = sm.add_constant(col[['año_centrado', 'es_travesti']])
modelo_travesti = sm.Logit(col['hubo_accion'], X).fit(disp=0)
print(modelo_travesti.summary())
print("\nADVERTENCIA: 'es_travesti' predice el resultado de forma casi perfecta")
print("(0 de 50 casos con acción institucional), lo que produce cuasi-separación:")
print("el modelo no converge y el coeficiente no tiene una estimación confiable.")
print("CONCLUSIÓN HONESTA para el artículo: no es estadísticamente posible aislar")
print("si la identidad 'travesti' en sí misma explica la ausencia de acción")
print("institucional, o si el patrón se debe enteramente a que estos casos")
print("ocurrieron en un período con menor documentación en general. Se reporta")
print("como hallazgo abierto, no como discriminación confirmada por identidad.")

                           Logit Regression Results                           
Dep. Variable:            hubo_accion   No. Observations:                  306
Model:                          Logit   Df Residuals:                      303
Method:                           MLE   Df Model:                            2
Date:                Mon, 31 Aug 2026   Pseudo R-squ.:                  0.3021
Time:                        03:23:44   Log-Likelihood:                -129.38
converged:                      False   LL-Null:                       -185.37
Covariance Type:            nonrobust   LLR p-value:                 4.814e-25
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -1.3921      0.225     -6.199      0.000      -1.832      -0.952
año_centrado     0.3314      0.051      6.544      0.000       0.232       0.431
es_travesti    -25.1511    1.7e+05     -0.00

/usr/local/lib/python3.13/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [5]:
# =========================================================
# MÉTODO 3 — TENDENCIA ANUAL: ¿el aumento de casos por año es
# estadísticamente significativo? (regresión de Poisson)
# =========================================================

# --- CELDA 5 ---
casos_año = col.groupby('Calendar year').size().reset_index(name='n')
casos_año['año_centrado'] = casos_año['Calendar year'] - casos_año['Calendar year'].mean()

X_poisson = sm.add_constant(casos_año['año_centrado'])
modelo_poisson = sm.GLM(casos_año['n'], X_poisson, family=sm.families.Poisson()).fit()
print(modelo_poisson.summary().tables[1])

razon_tasa = np.exp(modelo_poisson.params['año_centrado'])
print(f"\nRazón de tasa por año: {razon_tasa:.3f} "
      f"(la tasa de casos documentados sube ~{(razon_tasa-1)*100:.1f}% por año, en promedio)")
print("Estadísticamente significativo (p < 0.001), pero recuerda: esto mide")
print("casos DOCUMENTADOS, no necesariamente violencia real — ver Método 4.")

                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            2.7997      0.059     47.380      0.000       2.684       2.915
año_centrado     0.0501      0.011      4.455      0.000       0.028       0.072

Razón de tasa por año: 1.051 (la tasa de casos documentados sube ~5.1% por año, en promedio)
Estadísticamente significativo (p < 0.001), pero recuerda: esto mide
casos DOCUMENTADOS, no necesariamente violencia real — ver Método 4.


In [6]:
# =========================================================
# MÉTODO 4 — MODELO DE MISSINGNESS: ¿la probabilidad de que falte
# el dato de respuesta institucional cambia con el año? (evidencia
# formal de que el monitoreo mejoró, no solo intuición)
# =========================================================

# --- CELDA 6 ---
X_missing = sm.add_constant(col[['año_centrado']])
modelo_missing = sm.Logit(col['falta_dato_respuesta'], X_missing).fit(disp=0)
print(modelo_missing.summary().tables[1])

or_missing = np.exp(modelo_missing.params['año_centrado'])
print(f"\nPor cada año que pasa, la probabilidad de que falte el dato de respuesta")
print(f"institucional se multiplica por {or_missing:.3f} (estadísticamente significativo,")
print("p < 0.001). Es evidencia formal de que el seguimiento documental mejoró")
print("con el tiempo — apoya la lectura de que la Celda 5 mide mejor monitoreo,")
print("no solo más violencia.")

                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            1.4928      0.211      7.066      0.000       1.079       1.907
año_centrado    -0.3607      0.048     -7.532      0.000      -0.455      -0.267

Por cada año que pasa, la probabilidad de que falte el dato de respuesta
institucional se multiplica por 0.697 (estadísticamente significativo,
p < 0.001). Es evidencia formal de que el seguimiento documental mejoró
con el tiempo — apoya la lectura de que la Celda 5 mide mejor monitoreo,
no solo más violencia.


In [7]:
# =========================================================
# MÉTODO 5 — REGRESIÓN LOGÍSTICA SIN PROBLEMA DE SEPARACIÓN:
# ¿el año y ocurrir en una ciudad grande predicen alguna acción
# institucional?
# =========================================================

# --- CELDA 7 ---
X_final = sm.add_constant(col[['año_centrado', 'es_capital_grande']])
modelo_final = sm.Logit(col['hubo_accion'], X_final).fit(disp=0)
print(modelo_final.summary())

or_año = np.exp(modelo_final.params['año_centrado'])
or_capital = np.exp(modelo_final.params['es_capital_grande'])
print(f"\nPor cada año: la probabilidad de alguna acción institucional se multiplica")
print(f"por {or_año:.3f} (p={modelo_final.pvalues['año_centrado']:.4f}).")
print(f"Ocurrir en Bogotá/Medellín/Cali/Barranquilla vs. otro lugar: razón de")
print(f"momios = {or_capital:.3f} (p={modelo_final.pvalues['es_capital_grande']:.4f}).")

                           Logit Regression Results                           
Dep. Variable:            hubo_accion   No. Observations:                  306
Model:                          Logit   Df Residuals:                      303
Method:                           MLE   Df Model:                            2
Date:                Mon, 31 Aug 2026   Pseudo R-squ.:                  0.3042
Time:                        03:24:52   Log-Likelihood:                -128.98
converged:                       True   LL-Null:                       -185.37
Covariance Type:            nonrobust   LLR p-value:                 3.215e-25
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -1.3053      0.238     -5.484      0.000      -1.772      -0.839
año_centrado          0.3654      0.050      7.311      0.000       0.267       0.463
es_capital_grande    -0.

In [8]:
# --- CELDA 8: Exportar resultados ---
resultados = pd.DataFrame({
    'modelo': ['Fisher travesti vs mujer trans', 'Poisson tendencia anual',
               'Logit missingness ~ año', 'Logit acción ~ año + capital (año)',
               'Logit acción ~ año + capital (capital)'],
    'estadístico': [f'OR={odds:.3f}', f'RR={razon_tasa:.3f}', f'OR={or_missing:.3f}',
                     f'OR={or_año:.3f}', f'OR={or_capital:.3f}'],
    'p_valor': [p, modelo_poisson.pvalues['año_centrado'], modelo_missing.pvalues['año_centrado'],
                modelo_final.pvalues['año_centrado'], modelo_final.pvalues['es_capital_grande']]
})
print(resultados)
resultados.to_csv('/content/resultados_metodos_estadisticos.csv', index=False)
print("\nExportado: resultados_metodos_estadisticos.csv")

                                   modelo estadístico       p_valor
0          Fisher travesti vs mujer trans    OR=0.000  4.675424e-09
1                 Poisson tendencia anual    RR=1.051  8.407019e-06
2                 Logit missingness ~ año    OR=0.697  5.011607e-14
3      Logit acción ~ año + capital (año)    OR=1.441  2.652927e-13
4  Logit acción ~ año + capital (capital)    OR=0.462  2.037709e-02

Exportado: resultados_metodos_estadisticos.csv
